In [ ]:
# Cell 1: Load and inspect both sites
import pickle
import numpy as np
from pathlib import Path

SITE_A_DIR = Path(r"Z:\0_Thesis_Monzurul\Centerpoint_Road_Outlet_Mall")
SITE_B_DIR = Path(r"Z:\0_Thesis_Monzurul\Coldwater MI")

# ── Site A ───────────────────────────────────────────────────
with open(SITE_A_DIR / "sequences_stride4.pkl", "rb") as f:
    sequences_A, seq_labels_A = pickle.load(f)
seq_labels_A = np.array(seq_labels_A)

with open(SITE_A_DIR / "data_split.pkl", "rb") as f:
    split_A = pickle.load(f)

# ── Site B ───────────────────────────────────────────────────
with open(SITE_B_DIR / "sequences_stride4.pkl", "rb") as f:
    sequences_B, seq_labels_B = pickle.load(f)
seq_labels_B = np.array(seq_labels_B)

with open(SITE_B_DIR / "data_split.pkl", "rb") as f:
    split_B = pickle.load(f)

# ── Inspect ──────────────────────────────────────────────────
print("=== Site A ===")
print(f"Total sequences : {len(sequences_A):,}")
print(f"Conflict rate   : {seq_labels_A.mean():.1%}")
print(f"data_split keys : {list(split_A.keys())}")

print("\n=== Site B ===")
print(f"Total sequences : {len(sequences_B):,}")
print(f"Conflict rate   : {seq_labels_B.mean():.1%}")
print(f"data_split keys : {list(split_B.keys())}")

print("\n=== Site A split sizes ===")
for k, v in split_A.items():
    if hasattr(v, '__len__'):
        print(f"  {k:20s}: {len(v):,}", end="")
        if 'label' in k:
            print(f"  | conflict: {np.array(v).mean():.1%}", end="")
        print()

print("\n=== Site B split sizes ===")
for k, v in split_B.items():
    if hasattr(v, '__len__'):
        print(f"  {k:20s}: {len(v):,}", end="")
        if 'label' in k:
            print(f"  | conflict: {np.array(v).mean():.1%}", end="")
        print()

print("\n=== Sequence structure (Site A, seq 0, frame 0) ===")
g0 = sequences_A[0][0]
print(f"Graph keys  : {list(g0.keys())}")
print(f"video_id    : {g0['video_id']}")
print(f"num_nodes   : {g0['num_nodes']}")
n0 = g0["nodes"][0]
print(f"Node keys   : {list(n0.keys())}")
if g0["edges"]:
    e0 = g0["edges"][0]
    print(f"Edge keys   : {list(e0.keys())}")

In [ ]:
# Cell 2: Load norm stats and inspect model checkpoint
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Norm stats (Site A) ──────────────────────────────────────
with open(SITE_A_DIR / "norm_stats.pkl", "rb") as f:
    norm_A = pickle.load(f)

node_mean = norm_A["node_mean"]
node_std  = norm_A["node_std"]
edge_mean = norm_A["edge_mean"]
edge_std  = norm_A["edge_std"]

print(f"\nnorm_stats keys : {list(norm_A.keys())}")
print(f"node_mean shape : {node_mean.shape}  dtype: {node_mean.dtype}")
print(f"edge_mean shape : {edge_mean.shape}  dtype: {edge_mean.dtype}")

# ── Node/edge feature names for reference ────────────────────
NODE_FEATS = ["x", "y", "vx", "vy", "speed", "heading"]
EDGE_FEATS = ["distance", "rel_speed", "ttc", "drac", "dttc_dt", "heading_diff"]

print(f"\nnode_mean values:")
for i, name in enumerate(NODE_FEATS):
    if i < len(node_mean):
        print(f"  {name:12s}: mean={node_mean[i]:.4f}  std={node_std[i]:.4f}")

print(f"\nedge_mean values:")
for i, name in enumerate(EDGE_FEATS):
    if i < len(edge_mean):
        print(f"  {name:12s}: mean={edge_mean[i]:.4f}  std={edge_std[i]:.4f}")

# ── Inspect checkpoint ───────────────────────────────────────
print("\n=== Checkpoint files ===")
for fname in ["best_model.pt", "best_enhanced_model.pt"]:
    ckpt = SITE_A_DIR / fname
    if ckpt.exists():
        sd = torch.load(ckpt, map_location="cpu")
        print(f"\n{fname}")
        print(f"  Total params : {sum(v.numel() for v in sd.values()):,}")
        print(f"  Keys (all)   : {list(sd.keys())}")

In [ ]:
# Cell 2 addition — output directory (all saves go here, never to SITE_A_DIR)
SAVE_DIR = Path(r"Z:\0_Thesis_Monzurul\Coldwater MI\transfer_finetune")
SAVE_DIR.mkdir(exist_ok=True)
print(f"All outputs → {SAVE_DIR}")

In [ ]:
# Cell 3: Model definition + load checkpoint
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

NODE_DIM  = 9
EDGE_DIM  = 6
MAX_NODES = 9
SEQ_LEN   = 8

# ── Model classes ─────────────────────────────────────────────
class MultiHeadSSMAttention(nn.Module):
    def __init__(self, node_dim, hidden_dim, edge_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.W_q       = nn.Linear(node_dim,   hidden_dim)
        self.W_k       = nn.Linear(node_dim,   hidden_dim)
        self.W_v       = nn.Linear(node_dim,   hidden_dim)
        self.edge_enc  = nn.Linear(edge_dim,   num_heads)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim)
        self.leaky     = nn.LeakyReLU(0.2)

    def forward(self, X, A, E):
        B, N, _ = X.shape
        H = self.num_heads
        D = self.head_dim
        Q = self.W_q(X).view(B,N,H,D).permute(0,2,1,3)
        K = self.W_k(X).view(B,N,H,D).permute(0,2,1,3)
        V = self.W_v(X).view(B,N,H,D).permute(0,2,1,3)
        scale    = D ** -0.5
        scores   = torch.matmul(Q, K.transpose(-2,-1)) * scale
        ssm_bias = self.edge_enc(E).permute(0,3,1,2)
        scores   = scores + ssm_bias
        A_exp    = A.unsqueeze(1).expand(-1,H,-1,-1)
        scores   = scores.masked_fill(A_exp==0, -1e9)
        weights  = torch.softmax(scores, dim=-1)
        out = torch.matmul(weights, V)
        out = out.permute(0,2,1,3).contiguous().view(B, N, -1)
        return self.out_proj(out)


class EdgeUpdateLayer(nn.Module):
    def __init__(self, node_dim, edge_dim):
        super().__init__()
        self.edge_update = nn.Linear(node_dim + node_dim + edge_dim, edge_dim)
        self.norm = nn.LayerNorm(edge_dim)
        self.relu = nn.ReLU()

    def forward(self, X, E, A):
        B, N, _ = X.shape
        Xi = X.unsqueeze(2).expand(-1, -1, N, -1)
        Xj = X.unsqueeze(1).expand(-1, N, -1, -1)
        edge_input = torch.cat([Xi, Xj, E], dim=-1)
        E_new      = self.relu(self.edge_update(edge_input))
        mask       = A.unsqueeze(-1).expand_as(E)
        return self.norm(E + mask * (E_new - E))


class EnhancedHeteroGNN(nn.Module):
    def __init__(self, node_dim, hidden_dim, edge_dim, num_heads=4):
        super().__init__()
        self.input_proj    = nn.Linear(node_dim, hidden_dim)
        # Layer 1
        self.attn_vv_1     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_vp_1     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_pp_1     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.edge_update_1 = EdgeUpdateLayer(hidden_dim, edge_dim)
        self.fusion_1      = nn.Linear(hidden_dim * 4, hidden_dim)
        self.norm_1        = nn.LayerNorm(hidden_dim)
        # Layer 2
        self.attn_vv_2     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_vp_2     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_pp_2     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.edge_update_2 = EdgeUpdateLayer(hidden_dim, edge_dim)
        self.fusion_2      = nn.Linear(hidden_dim * 4, hidden_dim)
        self.norm_2        = nn.LayerNorm(hidden_dim)
        self.dropout       = nn.Dropout(0.3)

    def forward(self, X, A_vv, A_vp, A_pp, E):
        H = F.relu(self.input_proj(X))
        # Layer 1
        msg_vv = self.attn_vv_1(H, A_vv, E)
        msg_vp = self.attn_vp_1(H, A_vp, E)
        msg_pp = self.attn_pp_1(H, A_pp, E)
        H1 = self.norm_1(F.relu(
            self.fusion_1(torch.cat([H, msg_vv, msg_vp, msg_pp], dim=-1))
        )) + H
        A_combined = (A_vv + A_vp + A_pp).clamp(0, 1)
        E1 = self.edge_update_1(H1, E, A_combined)
        # Layer 2
        msg_vv2 = self.attn_vv_2(H1, A_vv, E1)
        msg_vp2 = self.attn_vp_2(H1, A_vp, E1)
        msg_pp2 = self.attn_pp_2(H1, A_pp, E1)
        H2 = self.norm_2(F.relu(
            self.fusion_2(torch.cat([H1, msg_vv2, msg_vp2, msg_pp2], dim=-1))
        )) + H1
        return self.dropout(H2), E1


class EnhancedSceneRiskModel(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden_dim=64,
                 edge_dim=EDGE_DIM, num_heads=4,
                 lstm_layers=2, dropout=0.3):
        super().__init__()
        self.gnn     = EnhancedHeteroGNN(node_dim, hidden_dim, edge_dim, num_heads)
        self.lstm    = nn.LSTM(hidden_dim, hidden_dim,
                               num_layers=lstm_layers, batch_first=True,
                               dropout=dropout if lstm_layers>1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_vv_seq, A_vp_seq, A_pp_seq, E_seq):
        B, T, N, _ = X_seq.shape
        frame_embeds = []
        for t in range(T):
            H, E_updated = self.gnn(
                X_seq[:,t], A_vv_seq[:,t],
                A_vp_seq[:,t], A_pp_seq[:,t], E_seq[:,t]
            )
            drac_scores  = E_updated[:,:,:,2].max(dim=-1).values
            risk_weights = torch.softmax(drac_scores, dim=-1).unsqueeze(-1)
            H_pool       = (H * risk_weights).sum(dim=1)
            frame_embeds.append(H_pool)
        H_seq  = torch.stack(frame_embeds, dim=1)
        out, _ = self.lstm(H_seq)
        final  = self.dropout(out[:,-1,:])
        return self.fc(final).squeeze(-1)


# ── Load checkpoint ───────────────────────────────────────────
base_model = EnhancedSceneRiskModel().to(DEVICE)
state_dict = torch.load(SITE_A_DIR / "best_enhanced_model.pt", map_location=DEVICE)
base_model.load_state_dict(state_dict)
base_model.eval()

# Freeze a copy — never modify this
BASE_STATE_DICT = copy.deepcopy(base_model.state_dict())

print(f"Model loaded successfully")
print(f"Parameters       : {sum(p.numel() for p in base_model.parameters()):,}")
print(f"Checkpoint params: {sum(v.numel() for v in state_dict.values()):,}")
print(f"Match            : {sum(p.numel() for p in base_model.parameters()) == sum(v.numel() for v in state_dict.values())}")

In [ ]:
# Quick check — run this first
g0 = split_B["train_seqs"][0][0]
print("Edge keys:", list(g0["edges"][0].keys()))

In [ ]:
# Cell 4: Dataset class + data loaders
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

NODE_DIM  = 9
EDGE_DIM  = 6
MAX_NODES = 9
SEQ_LEN   = 8
BATCH_SIZE = 16

# ── Exact function from 2_analysis.ipynb ─────────────────────
def scene_graph_to_tensors(graph, max_nodes,
                            node_mean, node_std,
                            edge_mean, edge_std):
    nodes  = graph["nodes"]
    edges  = graph["edges"]
    N      = len(nodes)
    id2idx = {n["track_id"]: i for i, n in enumerate(nodes)}

    X = []
    for n in nodes:
        raw    = np.array([
            n["x"], n["y"], n["vx"], n["vy"],
            n["speed"], n["accel"], n["heading"]
        ], dtype=np.float32)
        normed = (raw - node_mean) / node_std
        is_veh = 1.0 if n["agent_type"] == "vehicle"    else 0.0
        is_ped = 1.0 if n["agent_type"] == "pedestrian" else 0.0
        X.append(np.append(normed, [is_veh, is_ped]))
    X = np.array(X, dtype=np.float32)

    A_vv = np.zeros((N, N), dtype=np.float32)
    A_vp = np.zeros((N, N), dtype=np.float32)
    A_pp = np.zeros((N, N), dtype=np.float32)
    E    = np.zeros((N, N, EDGE_DIM), dtype=np.float32)

    for e in edges:
        i = id2idx.get(e["i"])
        j = id2idx.get(e["j"])
        if i is None or j is None:
            continue
        raw_ef = np.array([
            e["distance"],
            e["rel_speed"],
            e["drac"],
            e["heading_diff"],
            e["rel_vx"],
            e["rel_vy"]
        ], dtype=np.float32)
        normed_ef = (raw_ef - edge_mean) / edge_std
        E[i, j]   = normed_ef
        E[j, i]   = normed_ef
        itype = e["interaction_type"]
        if itype == "vv":
            A_vv[i,j] = A_vv[j,i] = 1
        elif itype == "vp":
            A_vp[i,j] = A_vp[j,i] = 1
        else:
            A_pp[i,j] = A_pp[j,i] = 1

    M        = max_nodes
    X_pad    = np.zeros((M, NODE_DIM),    dtype=np.float32)
    A_vv_pad = np.zeros((M, M),           dtype=np.float32)
    A_vp_pad = np.zeros((M, M),           dtype=np.float32)
    A_pp_pad = np.zeros((M, M),           dtype=np.float32)
    E_pad    = np.zeros((M, M, EDGE_DIM), dtype=np.float32)

    X_pad[:N]       = X
    A_vv_pad[:N,:N] = A_vv
    A_vp_pad[:N,:N] = A_vp
    A_pp_pad[:N,:N] = A_pp
    E_pad[:N,:N]    = E

    return X_pad, A_vv_pad, A_vp_pad, A_pp_pad, E_pad


# ── Exact class from 2_analysis.ipynb ────────────────────────
class SceneGraphDataset(Dataset):
    def __init__(self, sequences, labels,
                 node_mean, node_std,
                 edge_mean, edge_std,
                 max_nodes=MAX_NODES):
        self.sequences = sequences
        self.labels    = labels
        self.max_nodes = max_nodes
        self.node_mean = node_mean
        self.node_std  = node_std
        self.edge_mean = edge_mean
        self.edge_std  = edge_std

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq   = self.sequences[idx]
        label = self.labels[idx]
        X_seq, Avv_seq, Avp_seq, App_seq, E_seq = [], [], [], [], []
        for g in seq:
            X, Avv, Avp, App, E = scene_graph_to_tensors(
                g, self.max_nodes,
                self.node_mean, self.node_std,
                self.edge_mean, self.edge_std
            )
            X_seq.append(X)
            Avv_seq.append(Avv)
            Avp_seq.append(Avp)
            App_seq.append(App)
            E_seq.append(E)
        return (
            torch.tensor(np.stack(X_seq),   dtype=torch.float32),
            torch.tensor(np.stack(Avv_seq), dtype=torch.float32),
            torch.tensor(np.stack(Avp_seq), dtype=torch.float32),
            torch.tensor(np.stack(App_seq), dtype=torch.float32),
            torch.tensor(np.stack(E_seq),   dtype=torch.float32),
            torch.tensor(label,             dtype=torch.float32)
        )


# ── Build datasets ────────────────────────────────────────────
ds_A_train = SceneGraphDataset(
    split_A["train_seqs"], split_A["train_labels"],
    node_mean, node_std, edge_mean, edge_std
)
ds_A_val = SceneGraphDataset(
    split_A["val_seqs"], split_A["val_labels"],
    node_mean, node_std, edge_mean, edge_std
)
ds_B_train = SceneGraphDataset(
    split_B["train_seqs"], split_B["train_labels"],
    node_mean, node_std, edge_mean, edge_std
)
ds_B_val = SceneGraphDataset(
    split_B["val_seqs"], split_B["val_labels"],
    node_mean, node_std, edge_mean, edge_std
)
ds_B_test = SceneGraphDataset(
    split_B["test_seqs"], split_B["test_labels"],
    node_mean, node_std, edge_mean, edge_std
)

# Fixed loaders (never change)
loader_B_val  = DataLoader(ds_B_val,  batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0)
loader_B_test = DataLoader(ds_B_test, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0)

# ── Sanity check ──────────────────────────────────────────────
print(f"ds_A_train : {len(ds_A_train):,}")
print(f"ds_A_val   : {len(ds_A_val):,}")
print(f"ds_B_train : {len(ds_B_train):,}  | conflict: {np.array(split_B['train_labels']).mean():.1%}")
print(f"ds_B_val   : {len(ds_B_val):,}   | conflict: {np.array(split_B['val_labels']).mean():.1%}")
print(f"ds_B_test  : {len(ds_B_test):,}   | conflict: {np.array(split_B['test_labels']).mean():.1%}")

X, Avv, Avp, App, E, y = ds_B_train[0]
print(f"\nTensor shapes (one sample):")
print(f"  X   : {tuple(X.shape)}   ← [T, N, 9]")
print(f"  Avv : {tuple(Avv.shape)} ← [T, N, N]")
print(f"  E   : {tuple(E.shape)} ← [T, N, N, 6]")
print(f"  y   : {y.item()}")

In [ ]:
# Cell 5 (complete): Evaluate + fine-tune functions with full history

import copy
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss,
    cohen_kappa_score, matthews_corrcoef
)

# ── Evaluate ──────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for X, Avv, Avp, App, E, y in loader:
            X, Avv, Avp, App, E = (X.to(DEVICE), Avv.to(DEVICE),
                                    Avp.to(DEVICE), App.to(DEVICE),
                                    E.to(DEVICE))
            logits = model(X, Avv, Avp, App, E)
            y_true.extend(y.numpy())
            y_score.extend(torch.sigmoid(logits).cpu().numpy())

    y_true  = np.array(y_true)
    y_score = np.array(y_score)
    y_pred  = (y_score >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-8)
    spec = tn / max(tn + fp, 1)

    fpr_arr, tpr_arr, roc_thr = roc_curve(y_true, y_score)
    prec_arr, rec_arr, pr_thr = precision_recall_curve(y_true, y_score)

    def tpr_at_far(target):
        idx = np.clip(np.searchsorted(fpr_arr, target, side='right') - 1,
                      0, len(tpr_arr) - 1)
        return float(tpr_arr[idx])

    def far_at_tpr(target):
        idx = np.clip(np.searchsorted(tpr_arr, target, side='left'),
                      0, len(fpr_arr) - 1)
        return float(fpr_arr[idx])

    return {
        # ── Scalar metrics ───────────────────────────────────
        'auc_roc'     : float(roc_auc_score(y_true, y_score)),
        'auc_pr'      : float(average_precision_score(y_true, y_score)),
        'f1'          : float(f1),
        'precision'   : float(prec),
        'recall'      : float(rec),
        'specificity' : float(spec),
        'mcc'         : float(matthews_corrcoef(y_true, y_pred)),
        'kappa'       : float(cohen_kappa_score(y_true, y_pred)),
        'brier'       : float(brier_score_loss(y_true, y_score)),
        's_at_5pct'   : tpr_at_far(0.05),
        's_at_10pct'  : tpr_at_far(0.10),
        's_at_20pct'  : tpr_at_far(0.20),
        'far_at_80tpr': far_at_tpr(0.80),
        'far_at_90tpr': far_at_tpr(0.90),
        'far_at_95tpr': far_at_tpr(0.95),
        # ── Confusion matrix ─────────────────────────────────
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        # ── Curve arrays ─────────────────────────────────────
        'roc_fpr' : fpr_arr,
        'roc_tpr' : tpr_arr,
        'roc_thr' : roc_thr,
        'pr_prec' : prec_arr,
        'pr_rec'  : rec_arr,
        'pr_thr'  : pr_thr,
        'y_true'  : y_true,
        'y_score' : y_score,
    }


# ── Fine-tune with full history ───────────────────────────────
def finetune(base_state_dict, train_ds, val_loader,
             pos_w, lr=1e-4, epochs=30, patience=5):
    model = EnhancedSceneRiskModel().to(DEVICE)
    model.load_state_dict(copy.deepcopy(base_state_dict))

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_w]).to(DEVICE))
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0)

    best_auc, best_state, no_improve = 0.0, None, 0
    history = []

    for epoch in range(epochs):
        # ── Train ────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        n_batches  = 0
        for X, Avv, Avp, App, E, y in train_loader:
            X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                       Avp.to(DEVICE), App.to(DEVICE),
                                       E.to(DEVICE), y.to(DEVICE))
            optimizer.zero_grad()
            logits = model(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            n_batches  += 1
        train_loss /= max(n_batches, 1)

        # ── Validate ─────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        n_val    = 0
        y_true_v, y_score_v = [], []
        with torch.no_grad():
            for X, Avv, Avp, App, E, y in val_loader:
                X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                           Avp.to(DEVICE), App.to(DEVICE),
                                           E.to(DEVICE), y.to(DEVICE))
                logits   = model(X, Avv, Avp, App, E)
                loss     = criterion(logits, y)
                val_loss += loss.item()
                n_val    += 1
                y_true_v.extend(y.cpu().numpy())
                y_score_v.extend(torch.sigmoid(logits).cpu().numpy())

        val_loss  /= max(n_val, 1)
        y_true_v   = np.array(y_true_v)
        y_score_v  = np.array(y_score_v)
        y_pred_v   = (y_score_v >= 0.5).astype(int)

        val_auc   = float(roc_auc_score(y_true_v, y_score_v))
        val_pr    = float(average_precision_score(y_true_v, y_score_v))
        val_f1    = float(f1_score(y_true_v, y_pred_v, zero_division=0))
        val_prec  = float(precision_score(y_true_v, y_pred_v, zero_division=0))
        val_rec   = float(recall_score(y_true_v, y_pred_v, zero_division=0))
        val_brier = float(brier_score_loss(y_true_v, y_score_v))
        cm_v      = confusion_matrix(y_true_v, y_pred_v, labels=[0,1])
        tn_v, fp_v, fn_v, tp_v = cm_v.ravel()

        history.append({
            'epoch'        : epoch + 1,
            'train_loss'   : train_loss,
            'val_loss'     : val_loss,
            'val_auc_roc'  : val_auc,
            'val_auc_pr'   : val_pr,
            'val_f1'       : val_f1,
            'val_precision': val_prec,
            'val_recall'   : val_rec,
            'val_brier'    : val_brier,
            'val_tp'       : int(tp_v),
            'val_tn'       : int(tn_v),
            'val_fp'       : int(fp_v),
            'val_fn'       : int(fn_v),
        })

        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"    Early stop epoch {epoch+1} | "
                      f"best val AUC: {best_auc:.4f}")
                break

    model.load_state_dict(best_state)
    return model, history


# ── Zero-shot eval ────────────────────────────────────────────
print("Zero-shot eval (base model on Site B test)...")
zs_metrics = evaluate(base_model, loader_B_test)
print(f"  AUC-ROC  : {zs_metrics['auc_roc']:.4f}")
print(f"  AUC-PR   : {zs_metrics['auc_pr']:.4f}")
print(f"  F1       : {zs_metrics['f1']:.4f}")
print(f"  S@5%FAR  : {zs_metrics['s_at_5pct']:.4f}")
print(f"  S@10%FAR : {zs_metrics['s_at_10pct']:.4f}")
print(f"  Brier    : {zs_metrics['brier']:.4f}")
print(f"  TN={zs_metrics['tn']:,}  FP={zs_metrics['fp']:,}")
print(f"  FN={zs_metrics['fn']:,}  TP={zs_metrics['tp']:,}")

In [ ]:
# Check zero-shot confusion matrix in detail
print(f"Site B test total    : {5229}")
print(f"Site B test conflicts: {int(5229 * 0.058)} approx")
print(f"\nZero-shot CM:")
print(f"  TN={zs_metrics['tn']:,}  FP={zs_metrics['fp']:,}")
print(f"  FN={zs_metrics['fn']:,}  TP={zs_metrics['tp']:,}")
print(f"\nPrecision : {zs_metrics['precision']:.4f}")
print(f"Recall    : {zs_metrics['recall']:.4f}")
print(f"Specificity: {zs_metrics['specificity']:.4f}")

In [ ]:
# Cell 6 (complete): Fine-tuning loop with history + model saving

RATIOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SEEDS  = [42, 123, 456, 789, 2024]

train_seqs_B   = split_B["train_seqs"]
train_labels_B = np.array(split_B["train_labels"])
n_B_train      = len(train_seqs_B)

results = {}

# ── r=0%: zero-shot ───────────────────────────────────────────
results["B000"] = {
    'ratio'  : 0.0,
    'n_ft'   : 0,
    'seeds'  : [zs_metrics] * len(SEEDS),
    'history': [[] for _ in SEEDS],
    'mean'   : {k: v for k, v in zs_metrics.items()
                if isinstance(v, float)},
    'std'    : {k: 0.0 for k, v in zs_metrics.items()
                if isinstance(v, float)},
}
print(f"r=0% (zero-shot) stored | AUC-ROC: {zs_metrics['auc_roc']:.4f}")

# ── r=10% to 90%: fine-tune ───────────────────────────────────
for ratio in RATIOS[1:-1]:
    ratio_key      = f"B{int(ratio*100):03d}"
    n_ft           = max(1, int(round(ratio * n_B_train)))
    seed_metrics   = []
    seed_histories = []

    print(f"\n{'='*58}")
    print(f"  Ratio {ratio:.1f} ({int(ratio*100)}% Site B) | n_ft={n_ft:,}")
    print(f"{'='*58}")

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)
        rng = np.random.default_rng(seed)

        sampled_idx = rng.choice(n_B_train, size=n_ft, replace=False)
        ft_seqs     = [train_seqs_B[i] for i in sampled_idx]
        ft_labels   = train_labels_B[sampled_idx]

        n_pos = ft_labels.sum()
        n_neg = len(ft_labels) - n_pos
        pos_w = float(n_neg / max(n_pos, 1))

        ft_ds = SceneGraphDataset(
            ft_seqs, ft_labels,
            node_mean, node_std, edge_mean, edge_std
        )

        model, history = finetune(
            base_state_dict = BASE_STATE_DICT,
            train_ds        = ft_ds,
            val_loader      = loader_B_val,
            pos_w           = pos_w,
            lr              = 1e-4,
            epochs          = 30,
            patience        = 5,
        )

        metrics = evaluate(model, loader_B_test)
        seed_metrics.append(metrics)
        seed_histories.append(history)

        # ── Save checkpoint ───────────────────────────────────
        ckpt_name = f"model_r{int(ratio*100):03d}_s{seed}.pt"
        torch.save(model.state_dict(), SAVE_DIR / ckpt_name)

        print(f"  Seed {seed} | pos_w={pos_w:.1f} | "
              f"AUC-ROC: {metrics['auc_roc']:.4f} | "
              f"AUC-PR: {metrics['auc_pr']:.4f} | "
              f"F1: {metrics['f1']:.4f} | "
              f"S@5%: {metrics['s_at_5pct']:.4f} | "
              f"TP={metrics['tp']} FP={metrics['fp']} "
              f"TN={metrics['tn']} FN={metrics['fn']} | "
              f"saved: {ckpt_name}")

    float_keys = [k for k, v in seed_metrics[0].items() if isinstance(v, float)]
    results[ratio_key] = {
        'ratio'  : ratio,
        'n_ft'   : n_ft,
        'seeds'  : seed_metrics,
        'history': seed_histories,
        'mean'   : {k: float(np.mean([m[k] for m in seed_metrics])) for k in float_keys},
        'std'    : {k: float(np.std( [m[k] for m in seed_metrics])) for k in float_keys},
    }

    # ── Intermediate save after each ratio ────────────────────
    with open(SAVE_DIR / "finetune_results.pkl", "wb") as f:
        pickle.dump(results, f)
    print(f"  → Intermediate save done.")

# ── r=100%: upper bound — train from scratch ──────────────────
print(f"\n{'='*58}")
print(f"  Ratio 1.0 (100% Site B) — train from scratch")
print(f"{'='*58}")

seed_metrics   = []
seed_histories = []
n_pos = train_labels_B.sum()
n_neg = len(train_labels_B) - n_pos
pos_w = float(n_neg / max(n_pos, 1))

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    ft_ds = SceneGraphDataset(
        train_seqs_B, train_labels_B,
        node_mean, node_std, edge_mean, edge_std
    )

    model     = EnhancedSceneRiskModel().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_w]).to(DEVICE))
    optimizer = torch.optim.Adam(
        model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3)

    train_loader = DataLoader(ft_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0)

    best_auc, best_state, no_improve = 0.0, None, 0
    history = []

    for epoch in range(50):
        # ── Train ────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        n_batches  = 0
        for X, Avv, Avp, App, E, y in train_loader:
            X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                       Avp.to(DEVICE), App.to(DEVICE),
                                       E.to(DEVICE), y.to(DEVICE))
            optimizer.zero_grad()
            logits = model(X, Avv, Avp, App, E)
            loss   = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            n_batches  += 1
        train_loss /= max(n_batches, 1)

        # ── Validate ─────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        n_val    = 0
        y_true_v, y_score_v = [], []
        with torch.no_grad():
            for X, Avv, Avp, App, E, y in loader_B_val:
                X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                           Avp.to(DEVICE), App.to(DEVICE),
                                           E.to(DEVICE), y.to(DEVICE))
                logits   = model(X, Avv, Avp, App, E)
                loss     = criterion(logits, y)
                val_loss += loss.item()
                n_val    += 1
                y_true_v.extend(y.cpu().numpy())
                y_score_v.extend(torch.sigmoid(logits).cpu().numpy())

        val_loss  /= max(n_val, 1)
        y_true_v   = np.array(y_true_v)
        y_score_v  = np.array(y_score_v)
        y_pred_v   = (y_score_v >= 0.5).astype(int)

        val_auc   = float(roc_auc_score(y_true_v, y_score_v))
        val_pr    = float(average_precision_score(y_true_v, y_score_v))
        val_f1    = float(f1_score(y_true_v, y_pred_v, zero_division=0))
        val_prec  = float(precision_score(y_true_v, y_pred_v, zero_division=0))
        val_rec   = float(recall_score(y_true_v, y_pred_v, zero_division=0))
        val_brier = float(brier_score_loss(y_true_v, y_score_v))
        cm_v      = confusion_matrix(y_true_v, y_pred_v, labels=[0,1])
        tn_v, fp_v, fn_v, tp_v = cm_v.ravel()

        history.append({
            'epoch'        : epoch + 1,
            'train_loss'   : train_loss,
            'val_loss'     : val_loss,
            'val_auc_roc'  : val_auc,
            'val_auc_pr'   : val_pr,
            'val_f1'       : val_f1,
            'val_precision': val_prec,
            'val_recall'   : val_rec,
            'val_brier'    : val_brier,
            'val_tp'       : int(tp_v),
            'val_tn'       : int(tn_v),
            'val_fp'       : int(fp_v),
            'val_fn'       : int(fn_v),
        })

        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 5:
                print(f"    Early stop epoch {epoch+1} | "
                      f"best val AUC: {best_auc:.4f}")
                break

    model.load_state_dict(best_state)
    metrics = evaluate(model, loader_B_test)
    seed_metrics.append(metrics)
    seed_histories.append(history)

    # ── Save checkpoint ───────────────────────────────────────
    ckpt_name = f"model_r100_s{seed}.pt"
    torch.save(model.state_dict(), SAVE_DIR / ckpt_name)

    print(f"  Seed {seed} | pos_w={pos_w:.1f} | "
          f"AUC-ROC: {metrics['auc_roc']:.4f} | "
          f"AUC-PR: {metrics['auc_pr']:.4f} | "
          f"F1: {metrics['f1']:.4f} | "
          f"S@5%: {metrics['s_at_5pct']:.4f} | "
          f"TP={metrics['tp']} FP={metrics['fp']} "
          f"TN={metrics['tn']} FN={metrics['fn']} | "
          f"saved: {ckpt_name}")

float_keys = [k for k, v in seed_metrics[0].items() if isinstance(v, float)]
results["B100"] = {
    'ratio'  : 1.0,
    'n_ft'   : n_B_train,
    'seeds'  : seed_metrics,
    'history': seed_histories,
    'mean'   : {k: float(np.mean([m[k] for m in seed_metrics])) for k in float_keys},
    'std'    : {k: float(np.std( [m[k] for m in seed_metrics])) for k in float_keys},
}

# ── Final save ────────────────────────────────────────────────
with open(SAVE_DIR / "finetune_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"\nAll done. Results saved → {SAVE_DIR / 'finetune_results.pkl'}")
print(f"Model checkpoints  → {SAVE_DIR}")

In [ ]:
# Cell 7: Recover and save clean results up to ratio 0.9

import pickle

# Load the last intermediate save (contains B000 through B090)
with open(SAVE_DIR / "finetune_results.pkl", "rb") as f:
    results_recovered = pickle.load(f)

# Discard B100 if it partially crept in
results_recovered.pop("B100", None)

# Verify what we have
print("Keys present:", sorted(results_recovered.keys()))
for key in sorted(results_recovered.keys()):
    r = results_recovered[key]
    print(f"  {key} | ratio={r['ratio']} | n_ft={r['n_ft']:,} | "
          f"mean AUC-ROC={r['mean'].get('auc_roc', float('nan')):.4f}")

# Save clean version
with open(SAVE_DIR / "finetune_results_r000_to_r090.pkl", "wb") as f:
    pickle.dump(results_recovered, f)

print(f"\nClean results saved → {SAVE_DIR / 'finetune_results_r000_to_r090.pkl'}")
print(f"Total ratio levels  : {len(results_recovered)}")

In [ ]:
with open(SAVE_DIR / "finetune_results_r000_to_r090.pkl", "rb") as f:
    res = pickle.load(f)

# Check history content for one ratio/seed
sample_history = res["B090"]["history"][0]  # seed 42
print(type(sample_history))
if sample_history:
    print("History keys:", sample_history[0].keys())
    print("Num epochs recorded:", len(sample_history))

In [ ]:
# Cell 8: Mixed-data training (100% Site A + r% Site B segments → evaluate on Site B test)

import pickle, time

RATIOS = [0.1, 0.2, 0.3,0.4,0.5,0.6]
SEEDS  = [42, 123, 456]

train_seqs_A   = split_A["train_seqs"]
train_labels_A = np.array(split_A["train_labels"])
train_seqs_B   = split_B["train_seqs"]
train_labels_B = np.array(split_B["train_labels"])

train_vids_B  = np.array([seq[0]["video_id"] for seq in train_seqs_B])
unique_vids_B = np.unique(train_vids_B)
n_vids_B      = len(unique_vids_B)

print(f"Site A train sequences : {len(train_seqs_A):,}")
print(f"Site B train sequences : {len(train_seqs_B):,}")
print(f"Site B train segments  : {n_vids_B}")
print(f"Avg seqs/segment       : {len(train_seqs_B)/n_vids_B:.1f}")
print(f"Ratios to run          : {RATIOS}")
print(f"Seeds                  : {SEEDS}")
print(f"Total runs             : {len(RATIOS) * len(SEEDS)}")

mixed_results = {}
run_counter   = 0
total_runs    = len(RATIOS) * len(SEEDS)

for ratio_idx, ratio in enumerate(RATIOS):
    ratio_key  = f"M{int(ratio*100):03d}"
    n_vids_sel = max(1, int(round(ratio * n_vids_B)))
    seed_metrics   = []
    seed_histories = []

    ratio_start = time.time()
    print(f"\n{'='*62}")
    print(f"  RATIO {ratio_idx+1}/{len(RATIOS)} | {int(ratio*100)}% Site B "
          f"| {n_vids_sel}/{n_vids_B} segments | key={ratio_key}")
    print(f"{'='*62}")

    for seed_idx, seed in enumerate(SEEDS):
        run_counter += 1
        seed_start   = time.time()

        print(f"\n  ── Seed {seed} ({seed_idx+1}/{len(SEEDS)}) "
              f"| overall run {run_counter}/{total_runs} ──")

        torch.manual_seed(seed)
        np.random.seed(seed)
        rng = np.random.default_rng(seed)

        selected_vids = rng.choice(unique_vids_B, size=n_vids_sel, replace=False)
        selected_set  = set(selected_vids)

        sel_mask    = np.array([vid in selected_set for vid in train_vids_B])
        ft_seqs_B   = [train_seqs_B[i] for i in np.where(sel_mask)[0]]
        ft_labels_B = train_labels_B[sel_mask]

        mixed_seqs   = list(train_seqs_A) + list(ft_seqs_B)
        mixed_labels = np.concatenate([train_labels_A, ft_labels_B])

        n_pos = mixed_labels.sum()
        n_neg = len(mixed_labels) - n_pos
        pos_w = float(n_neg / max(n_pos, 1))

        print(f"     Dataset : {len(mixed_seqs):,} seqs "
              f"(A={len(train_seqs_A):,} + B={len(ft_seqs_B):,}) | "
              f"pos={int(n_pos)} neg={int(n_neg)} pos_w={pos_w:.1f}")

        mixed_ds = SceneGraphDataset(
            mixed_seqs, mixed_labels,
            node_mean, node_std, edge_mean, edge_std
        )

        model     = EnhancedSceneRiskModel().to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_w]).to(DEVICE))
        optimizer = torch.optim.Adam(
            model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=3)

        train_loader = DataLoader(mixed_ds, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=0)

        best_auc, best_state, no_improve = 0.0, None, 0
        history = []

        print(f"     Training (max 30 epochs, patience=5) ...")

        for epoch in range(30):
            # ── Train ─────────────────────────────────────────
            model.train()
            train_loss, n_batches = 0.0, 0
            for X, Avv, Avp, App, E, y in train_loader:
                X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                           Avp.to(DEVICE), App.to(DEVICE),
                                           E.to(DEVICE), y.to(DEVICE))
                optimizer.zero_grad()
                logits = model(X, Avv, Avp, App, E)
                loss   = criterion(logits, y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_loss += loss.item()
                n_batches  += 1
            train_loss /= max(n_batches, 1)

            # ── Validate ──────────────────────────────────────
            model.eval()
            val_loss, n_val = 0.0, 0
            y_true_v, y_score_v = [], []
            with torch.no_grad():
                for X, Avv, Avp, App, E, y in loader_B_val:
                    X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                               Avp.to(DEVICE), App.to(DEVICE),
                                               E.to(DEVICE), y.to(DEVICE))
                    logits   = model(X, Avv, Avp, App, E)
                    loss     = criterion(logits, y)
                    val_loss += loss.item()
                    n_val    += 1
                    y_true_v.extend(y.cpu().numpy())
                    y_score_v.extend(torch.sigmoid(logits).cpu().numpy())

            val_loss  /= max(n_val, 1)
            y_true_v   = np.array(y_true_v)
            y_score_v  = np.array(y_score_v)
            y_pred_v   = (y_score_v >= 0.5).astype(int)

            val_auc   = float(roc_auc_score(y_true_v, y_score_v))
            val_pr    = float(average_precision_score(y_true_v, y_score_v))
            val_f1    = float(f1_score(y_true_v, y_pred_v, zero_division=0))
            val_prec  = float(precision_score(y_true_v, y_pred_v, zero_division=0))
            val_rec   = float(recall_score(y_true_v, y_pred_v, zero_division=0))
            val_brier = float(brier_score_loss(y_true_v, y_score_v))
            cm_v      = confusion_matrix(y_true_v, y_pred_v, labels=[0,1])
            tn_v, fp_v, fn_v, tp_v = cm_v.ravel()

            improved = val_auc > best_auc
            flag     = " ✓ best" if improved else f" (no improve {no_improve+1}/5)"

            # ── Per-epoch print ───────────────────────────────
            print(f"     Ep {epoch+1:02d} | "
                  f"train_loss={train_loss:.4f} | "
                  f"val_loss={val_loss:.4f} | "
                  f"val_AUC={val_auc:.4f} | "
                  f"val_F1={val_f1:.4f}{flag}")

            history.append({
                'epoch'        : epoch + 1,
                'train_loss'   : train_loss,
                'val_loss'     : val_loss,
                'val_auc_roc'  : val_auc,
                'val_auc_pr'   : val_pr,
                'val_f1'       : val_f1,
                'val_precision': val_prec,
                'val_recall'   : val_rec,
                'val_brier'    : val_brier,
                'val_tp'       : int(tp_v),
                'val_tn'       : int(tn_v),
                'val_fp'       : int(fp_v),
                'val_fn'       : int(fn_v),
            })

            scheduler.step(val_auc)

            if improved:
                best_auc   = val_auc
                best_state = copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= 5:
                    print(f"     → Early stop at epoch {epoch+1} | "
                          f"best val AUC: {best_auc:.4f}")
                    break

        model.load_state_dict(best_state)
        metrics = evaluate(model, loader_B_test)
        seed_metrics.append(metrics)
        seed_histories.append(history)

        ckpt_name = f"mixed_r{int(ratio*100):03d}_s{seed}.pt"
        torch.save(model.state_dict(), SAVE_DIR / ckpt_name)

        seed_elapsed = time.time() - seed_start
        print(f"     ✔ Seed {seed} done in {seed_elapsed/60:.1f} min | "
              f"AUC-ROC={metrics['auc_roc']:.4f} | "
              f"AUC-PR={metrics['auc_pr']:.4f} | "
              f"F1={metrics['f1']:.4f} | "
              f"S@5%={metrics['s_at_5pct']:.4f} | "
              f"TP={metrics['tp']} FP={metrics['fp']} "
              f"TN={metrics['tn']} FN={metrics['fn']} | "
              f"saved: {ckpt_name}")

    float_keys = [k for k, v in seed_metrics[0].items() if isinstance(v, float)]
    mixed_results[ratio_key] = {
        'ratio'     : ratio,
        'n_vids_sel': n_vids_sel,
        'n_B_seqs'  : int(sel_mask.sum()),
        'n_A_seqs'  : len(train_seqs_A),
        'n_mixed'   : len(mixed_seqs),
        'seeds'     : seed_metrics,
        'history'   : seed_histories,
        'mean'      : {k: float(np.mean([m[k] for m in seed_metrics])) for k in float_keys},
        'std'       : {k: float(np.std( [m[k] for m in seed_metrics])) for k in float_keys},
    }

    with open(SAVE_DIR / "mixed_results.pkl", "wb") as f:
        pickle.dump(mixed_results, f)

    ratio_elapsed = time.time() - ratio_start
    print(f"\n  → Ratio {int(ratio*100)}% complete in {ratio_elapsed/60:.1f} min | "
          f"mean AUC-ROC={mixed_results[ratio_key]['mean']['auc_roc']:.4f} ± "
          f"{mixed_results[ratio_key]['std']['auc_roc']:.4f} | "
          f"intermediate save done.")

print(f"\nAll done → {SAVE_DIR / 'mixed_results.pkl'}")

# extra

In [ ]:
class EnhancedHeteroGNN(nn.Module):
    def __init__(self, node_dim, hidden_dim, edge_dim, num_heads=4):
        super().__init__()
        self.input_proj    = nn.Linear(node_dim, hidden_dim)
        # Layer 1
        self.attn_vv_1     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_vp_1     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_pp_1     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.edge_update_1 = EdgeUpdateLayer(hidden_dim, edge_dim)
        self.fusion_1      = nn.Linear(hidden_dim * 4, hidden_dim)
        self.norm_1        = nn.LayerNorm(hidden_dim)
        # Layer 2
        self.attn_vv_2     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_vp_2     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.attn_pp_2     = MultiHeadSSMAttention(hidden_dim, hidden_dim, edge_dim, num_heads)
        self.edge_update_2 = EdgeUpdateLayer(hidden_dim, edge_dim)
        self.fusion_2      = nn.Linear(hidden_dim * 4, hidden_dim)
        self.norm_2        = nn.LayerNorm(hidden_dim)
        self.dropout       = nn.Dropout(0.3)

    def forward(self, X, A_vv, A_vp, A_pp, E):
        H = F.relu(self.input_proj(X))
        # Layer 1
        msg_vv = self.attn_vv_1(H, A_vv, E)
        msg_vp = self.attn_vp_1(H, A_vp, E)
        msg_pp = self.attn_pp_1(H, A_pp, E)
        H1 = self.norm_1(F.relu(
            self.fusion_1(torch.cat([H, msg_vv, msg_vp, msg_pp], dim=-1))
        )) + H
        A_combined = (A_vv + A_vp + A_pp).clamp(0, 1)
        E1 = self.edge_update_1(H1, E, A_combined)
        # Layer 2
        msg_vv2 = self.attn_vv_2(H1, A_vv, E1)
        msg_vp2 = self.attn_vp_2(H1, A_vp, E1)
        msg_pp2 = self.attn_pp_2(H1, A_pp, E1)
        H2 = self.norm_2(F.relu(
            self.fusion_2(torch.cat([H1, msg_vv2, msg_vp2, msg_pp2], dim=-1))
        )) + H1
        E2 = self.edge_update_2(H2, E1, A_combined)
        return self.dropout(H2), E2

In [ ]:
# Cell 2 addition — output directory (all saves go here, never to SITE_A_DIR)
SAVE_DIR = Path(r"Z:\0_Thesis_Monzurul\Coldwater MI\transfer_finetune2")
SAVE_DIR.mkdir(exist_ok=True)
print(f"All outputs → {SAVE_DIR}")

In [ ]:
# Cell 8: Mixed-data training (100% Site A + r% Site B segments → evaluate on Site B test)

import pickle, time

RATIOS = [0.1, 0.2, 0.3,0.4,0.5,0.6]
SEEDS  = [1112]

train_seqs_A   = split_A["train_seqs"]
train_labels_A = np.array(split_A["train_labels"])
train_seqs_B   = split_B["train_seqs"]
train_labels_B = np.array(split_B["train_labels"])

train_vids_B  = np.array([seq[0]["video_id"] for seq in train_seqs_B])
unique_vids_B = np.unique(train_vids_B)
n_vids_B      = len(unique_vids_B)

print(f"Site A train sequences : {len(train_seqs_A):,}")
print(f"Site B train sequences : {len(train_seqs_B):,}")
print(f"Site B train segments  : {n_vids_B}")
print(f"Avg seqs/segment       : {len(train_seqs_B)/n_vids_B:.1f}")
print(f"Ratios to run          : {RATIOS}")
print(f"Seeds                  : {SEEDS}")
print(f"Total runs             : {len(RATIOS) * len(SEEDS)}")

mixed_results = {}
run_counter   = 0
total_runs    = len(RATIOS) * len(SEEDS)

for ratio_idx, ratio in enumerate(RATIOS):
    ratio_key  = f"M{int(ratio*100):03d}"
    n_vids_sel = max(1, int(round(ratio * n_vids_B)))
    seed_metrics   = []
    seed_histories = []

    ratio_start = time.time()
    print(f"\n{'='*62}")
    print(f"  RATIO {ratio_idx+1}/{len(RATIOS)} | {int(ratio*100)}% Site B "
          f"| {n_vids_sel}/{n_vids_B} segments | key={ratio_key}")
    print(f"{'='*62}")

    for seed_idx, seed in enumerate(SEEDS):
        run_counter += 1
        seed_start   = time.time()

        print(f"\n  ── Seed {seed} ({seed_idx+1}/{len(SEEDS)}) "
              f"| overall run {run_counter}/{total_runs} ──")

        torch.manual_seed(seed)
        np.random.seed(seed)
        rng = np.random.default_rng(seed)

        selected_vids = rng.choice(unique_vids_B, size=n_vids_sel, replace=False)
        selected_set  = set(selected_vids)

        sel_mask    = np.array([vid in selected_set for vid in train_vids_B])
        ft_seqs_B   = [train_seqs_B[i] for i in np.where(sel_mask)[0]]
        ft_labels_B = train_labels_B[sel_mask]

        mixed_seqs   = list(train_seqs_A) + list(ft_seqs_B)
        mixed_labels = np.concatenate([train_labels_A, ft_labels_B])

        n_pos = mixed_labels.sum()
        n_neg = len(mixed_labels) - n_pos
        pos_w = float(n_neg / max(n_pos, 1))

        print(f"     Dataset : {len(mixed_seqs):,} seqs "
              f"(A={len(train_seqs_A):,} + B={len(ft_seqs_B):,}) | "
              f"pos={int(n_pos)} neg={int(n_neg)} pos_w={pos_w:.1f}")

        mixed_ds = SceneGraphDataset(
            mixed_seqs, mixed_labels,
            node_mean, node_std, edge_mean, edge_std
        )

        model     = EnhancedSceneRiskModel().to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_w]).to(DEVICE))
        optimizer = torch.optim.Adam(
            model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=3)

        train_loader = DataLoader(mixed_ds, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=0)

        best_auc, best_state, no_improve = 0.0, None, 0
        history = []

        print(f"     Training (max 30 epochs, patience=5) ...")

        for epoch in range(30):
            # ── Train ─────────────────────────────────────────
            model.train()
            train_loss, n_batches = 0.0, 0
            for X, Avv, Avp, App, E, y in train_loader:
                X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                           Avp.to(DEVICE), App.to(DEVICE),
                                           E.to(DEVICE), y.to(DEVICE))
                optimizer.zero_grad()
                logits = model(X, Avv, Avp, App, E)
                loss   = criterion(logits, y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_loss += loss.item()
                n_batches  += 1
            train_loss /= max(n_batches, 1)

            # ── Validate ──────────────────────────────────────
            model.eval()
            val_loss, n_val = 0.0, 0
            y_true_v, y_score_v = [], []
            with torch.no_grad():
                for X, Avv, Avp, App, E, y in loader_B_val:
                    X, Avv, Avp, App, E, y = (X.to(DEVICE), Avv.to(DEVICE),
                                               Avp.to(DEVICE), App.to(DEVICE),
                                               E.to(DEVICE), y.to(DEVICE))
                    logits   = model(X, Avv, Avp, App, E)
                    loss     = criterion(logits, y)
                    val_loss += loss.item()
                    n_val    += 1
                    y_true_v.extend(y.cpu().numpy())
                    y_score_v.extend(torch.sigmoid(logits).cpu().numpy())

            val_loss  /= max(n_val, 1)
            y_true_v   = np.array(y_true_v)
            y_score_v  = np.array(y_score_v)
            y_pred_v   = (y_score_v >= 0.5).astype(int)

            val_auc   = float(roc_auc_score(y_true_v, y_score_v))
            val_pr    = float(average_precision_score(y_true_v, y_score_v))
            val_f1    = float(f1_score(y_true_v, y_pred_v, zero_division=0))
            val_prec  = float(precision_score(y_true_v, y_pred_v, zero_division=0))
            val_rec   = float(recall_score(y_true_v, y_pred_v, zero_division=0))
            val_brier = float(brier_score_loss(y_true_v, y_score_v))
            cm_v      = confusion_matrix(y_true_v, y_pred_v, labels=[0,1])
            tn_v, fp_v, fn_v, tp_v = cm_v.ravel()

            improved = val_auc > best_auc
            flag     = " ✓ best" if improved else f" (no improve {no_improve+1}/5)"

            # ── Per-epoch print ───────────────────────────────
            print(f"     Ep {epoch+1:02d} | "
                  f"train_loss={train_loss:.4f} | "
                  f"val_loss={val_loss:.4f} | "
                  f"val_AUC={val_auc:.4f} | "
                  f"val_F1={val_f1:.4f}{flag}")

            history.append({
                'epoch'        : epoch + 1,
                'train_loss'   : train_loss,
                'val_loss'     : val_loss,
                'val_auc_roc'  : val_auc,
                'val_auc_pr'   : val_pr,
                'val_f1'       : val_f1,
                'val_precision': val_prec,
                'val_recall'   : val_rec,
                'val_brier'    : val_brier,
                'val_tp'       : int(tp_v),
                'val_tn'       : int(tn_v),
                'val_fp'       : int(fp_v),
                'val_fn'       : int(fn_v),
            })

            scheduler.step(val_auc)

            if improved:
                best_auc   = val_auc
                best_state = copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= 5:
                    print(f"     → Early stop at epoch {epoch+1} | "
                          f"best val AUC: {best_auc:.4f}")
                    break

        model.load_state_dict(best_state)
        metrics = evaluate(model, loader_B_test)
        seed_metrics.append(metrics)
        seed_histories.append(history)

        ckpt_name = f"mixed_r{int(ratio*100):03d}_s{seed}.pt"
        torch.save(model.state_dict(), SAVE_DIR / ckpt_name)

        seed_elapsed = time.time() - seed_start
        print(f"     ✔ Seed {seed} done in {seed_elapsed/60:.1f} min | "
              f"AUC-ROC={metrics['auc_roc']:.4f} | "
              f"AUC-PR={metrics['auc_pr']:.4f} | "
              f"F1={metrics['f1']:.4f} | "
              f"S@5%={metrics['s_at_5pct']:.4f} | "
              f"TP={metrics['tp']} FP={metrics['fp']} "
              f"TN={metrics['tn']} FN={metrics['fn']} | "
              f"saved: {ckpt_name}")

    float_keys = [k for k, v in seed_metrics[0].items() if isinstance(v, float)]
    mixed_results[ratio_key] = {
        'ratio'     : ratio,
        'n_vids_sel': n_vids_sel,
        'n_B_seqs'  : int(sel_mask.sum()),
        'n_A_seqs'  : len(train_seqs_A),
        'n_mixed'   : len(mixed_seqs),
        'seeds'     : seed_metrics,
        'history'   : seed_histories,
        'mean'      : {k: float(np.mean([m[k] for m in seed_metrics])) for k in float_keys},
        'std'       : {k: float(np.std( [m[k] for m in seed_metrics])) for k in float_keys},
    }

    with open(SAVE_DIR / "mixed_results.pkl", "wb") as f:
        pickle.dump(mixed_results, f)

    ratio_elapsed = time.time() - ratio_start
    print(f"\n  → Ratio {int(ratio*100)}% complete in {ratio_elapsed/60:.1f} min | "
          f"mean AUC-ROC={mixed_results[ratio_key]['mean']['auc_roc']:.4f} ± "
          f"{mixed_results[ratio_key]['std']['auc_roc']:.4f} | "
          f"intermediate save done.")

print(f"\nAll done → {SAVE_DIR / 'mixed_results.pkl'}")

In [ ]:
# Cell 2 addition — output directory (all saves go here, never to SITE_A_DIR)
SAVE_DIR = Path(r"Z:\0_Thesis_Monzurul\Coldwater MI\transfer_finetune")
SAVE_DIR.mkdir(exist_ok=True)
print(f"All outputs → {SAVE_DIR}")

In [ ]:
# Cell 9: Plot loss curves, metrics table, and confusion matrices

import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load saved results
with open(SAVE_DIR / "mixed_results.pkl", "rb") as f:
    mixed_results = pickle.load(f)

# -----------------------------
# 1. Loss curve
# -----------------------------
plt.figure(figsize=(10, 6))

for ratio_key, result in mixed_results.items():
    ratio = int(result["ratio"] * 100)
    history = result["history"][0]   # seed 1112

    hist_df = pd.DataFrame(history)

    plt.plot(
        hist_df["epoch"],
        hist_df["train_loss"],
        marker="o",
        label=f"{ratio}% B Train Loss"
    )

    plt.plot(
        hist_df["epoch"],
        hist_df["val_loss"],
        marker="s",
        linestyle="--",
        label=f"{ratio}% B Val Loss"
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / "loss_curves.png")
plt.show()


# -----------------------------
# 2. All metrics in one table
# -----------------------------
rows = []

for ratio_key, result in mixed_results.items():
    ratio = int(result["ratio"] * 100)
    metrics = result["seeds"][0]

    row = {
        "Ratio": f"{ratio}%",
        "AUC-ROC": metrics.get("auc_roc"),
        "AUC-PR": metrics.get("auc_pr"),
        "F1": metrics.get("f1"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "Brier": metrics.get("brier"),
        "S@5%": metrics.get("s_at_5pct"),
        "TP": metrics.get("tp"),
        "FP": metrics.get("fp"),
        "TN": metrics.get("tn"),
        "FN": metrics.get("fn"),
    }

    rows.append(row)

metrics_df = pd.DataFrame(rows)

display(metrics_df.round(4))


# Optional: save table
metrics_df.to_csv(SAVE_DIR / "mixed_metrics_summary.csv", index=False)


# -----------------------------
# 3. Confusion matrices
# -----------------------------
for ratio_key, result in mixed_results.items():
    ratio = int(result["ratio"] * 100)
    metrics = result["seeds"][0]

    cm = [
        [metrics["tn"], metrics["fp"]],
        [metrics["fn"], metrics["tp"]]
    ]

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"]
    )

    plt.title(f"Confusion Matrix - {ratio}% Site B")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(SAVE_DIR / f"confusion_matrix_{ratio_key}.png")
    plt.show()

In [ ]:
# -----------------------------
# 1. Separate train and validation loss curves
#    y-axis fixed from 0 to 1
# -----------------------------

# Train loss curve
plt.figure(figsize=(10, 6))

for ratio_key, result in mixed_results.items():
    ratio = int(result["ratio"] * 100)
    history = result["history"][0]
    hist_df = pd.DataFrame(history)

    plt.plot(
        hist_df["epoch"],
        hist_df["train_loss"],
        marker="o",
        label=f"{ratio}% Site B"
    )

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Curve")
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / "train_loss_curves.png")
plt.show()


# Validation loss curve
plt.figure(figsize=(10, 6))

for ratio_key, result in mixed_results.items():
    ratio = int(result["ratio"] * 100)
    history = result["history"][0]
    hist_df = pd.DataFrame(history)

    plt.plot(
        hist_df["epoch"],
        hist_df["val_loss"],
        marker="s",
        linestyle="--",
        label=f"{ratio}% Site B"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss Curve")
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR / "val_loss_curves.png")
plt.show()

In [ ]:
# -----------------------------
# 4. Plot all metrics in one figure
# -----------------------------

# Accuracy may not exist directly, so calculate from confusion matrix
metrics_df["Accuracy"] = (
    (metrics_df["TP"] + metrics_df["TN"]) /
    (metrics_df["TP"] + metrics_df["TN"] + metrics_df["FP"] + metrics_df["FN"])
)

plot_cols = [
    "Accuracy",
    "AUC-ROC",
    "AUC-PR",
    "F1",
    "Precision",
    "Recall",
    "Brier",
    "S@5%"
]

plt.figure(figsize=(12, 7))

for col in plot_cols:
    plt.plot(
        metrics_df["Ratio"],
        metrics_df[col],
        marker="o",
        linewidth=2,
        label=col
    )

plt.xlabel("Site B Training Ratio")
plt.ylabel("Metric Value")
plt.title("Model Performance Metrics Across Site B Training Ratios")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend(loc="best", ncol=2)
plt.tight_layout()
plt.savefig(SAVE_DIR / "all_metrics_comparison.png")
plt.show()

In [ ]:
# Print train loss and validation loss data in table

loss_rows = []

for ratio_key, result in mixed_results.items():
    ratio = int(result["ratio"] * 100)
    history = result["history"][0]   # seed 1112

    for h in history:
        loss_rows.append({
            "Ratio": f"{ratio}%",
            "Epoch": h["epoch"],
            "Train Loss": h["train_loss"],
            "Validation Loss": h["val_loss"]
        })

loss_df = pd.DataFrame(loss_rows)

display(loss_df.round(4))

# Optional: save as CSV
loss_df.to_csv(SAVE_DIR / "train_val_loss_table.csv", index=False)